In [0]:
%run ../utils

In [0]:
import os
import json
import requests
from pydantic import BaseModel, Field

In [0]:
dbutils.widgets.text("base_llm_model", "databricks-meta-llama-3-3-70b-instruct")
base_llm_model = dbutils.widgets.get("base_llm_model")
print("Base model:", base_llm_model)
#ananya-rac-gpt4o-ep
#databricks-llama-4-maverick

In [0]:
client = get_client()

In [0]:
# --------------------------------------------------------------
# Define the knowledge base retrieval tool
# --------------------------------------------------------------


def search_kb(question: str):
    """
    Load the whole knowledge base from the JSON file.
    (This is a mock function for demonstration purposes, we don't search)
    """
    with open("kb.json", "r") as f:
        return json.load(f)

In [0]:
# --------------------------------------------------------------
# Step 1: Call model with search_kb tool defined
# --------------------------------------------------------------

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_kb",
            "description": "Get the answer to the user's question from the knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                },
                "required": ["question"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

system_prompt = "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the return policy?"},
]


In [0]:
completion = get_completion_create(
  oai_client= client,
  model=base_llm_model,
  messages=messages,
  tools=tools
)

In [0]:
# --------------------------------------------------------------
# Step 2: Model decides to call function(s)
# --------------------------------------------------------------

completion.model_dump()

In [0]:
completion.choices[0]

In [0]:
completion.choices[0].message.tool_calls

In [0]:
# --------------------------------------------------------------
# Step 3: Execute search_kb function
# --------------------------------------------------------------


def call_function(name, args):
    if name == "search_kb":
        return search_kb(**args)


for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    messages.append(completion.choices[0].message)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [0]:
messages

In [0]:
# --------------------------------------------------------------
# Step 4: Supply result and call model again
# --------------------------------------------------------------


class KBResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    source: int = Field(description="The record id of the answer.")



completion_2 = get_completion_parse(
  oai_client=client,
  model=base_llm_model,
  messages=messages,
  tools=tools,
  response_format=KBResponse
)

In [0]:
# --------------------------------------------------------------
# Step 5: Check model response
# --------------------------------------------------------------

final_response = completion_2.choices[0].message.parsed
print(final_response.answer)
print(final_response.source)

In [0]:
# --------------------------------------------------------------
# Question that doesn't trigger the tool
# --------------------------------------------------------------

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the weather in Tokyo?"},
]

completion_3 = get_completion_parse(
  oai_client=client,
  model=base_llm_model,
  messages=messages,
  tools=tools
)

completion_3.choices[0].message

In [0]:
completion_3.choices[0].message.content